# α-AGI Governance · Review before release

Run all cells on Python 3.11–3.13. The maintained workbench uses only the standard library. No API key, GPU or wallet is required. All five cases are constructed examples; their inputs need independent review. Passing a model does not approve execution.

[Browser workbench](https://montrealai.github.io/AGI-Alpha-Agent-v0/solving_agi_governance/) · [Original notebook archive](research_notebook_archive.ipynb)

## 1. Open reproducible source

The default is release v1.14.0. Fetching public source needs a network once. Set `GOVERNANCE_SOURCE` to an existing checkout for local/offline use; no existing directory is overwritten. The exact source commit is printed.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use a Python 3.11, 3.12 or 3.13 runtime")
release = "v1.14.0"
configured_source = os.environ.get("GOVERNANCE_SOURCE")
source = Path(configured_source).expanduser().resolve() if configured_source else Path.cwd() / "governance-v1.14.0"
if not source.exists():
    if configured_source:
        raise FileNotFoundError("GOVERNANCE_SOURCE does not exist; choose an existing checkout")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", release,
                    "https://github.com/MontrealAI/AGI-Alpha-Agent-v0.git", str(source)], check=True)
module = source / "alpha_factory_v1/demos/solving_agi_governance/workbench.py"
if not module.is_file():
    raise RuntimeError("This directory does not contain the maintained governance workbench")
if not configured_source:
    checked_tag = subprocess.check_output(
        ["git", "-C", str(source), "describe", "--exact-match", "--tags"], text=True).strip()
    if checked_tag != release:
        raise RuntimeError("The existing source directory is not the expected release")
commit = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", commit)
sys.path.insert(0, str(source))
from alpha_factory_v1.demos.solving_agi_governance.workbench import read_json, evaluate, verify, write_bundle, brief
cases = read_json(module.with_name("scenarios.json"))
print("Cases:", ", ".join(case["id"] for case in cases))

## 2. Inspect every case

Only the accountable upgrade passes all modeled gates. The remaining cases illustrate shared control, incentive failure, aggregate risk and operational holds.

In [ ]:
reports = [evaluate(case) for case in cases]
for report in reports:
    failed = [g["id"] for g in report["result"]["gates"] if not g["passed"]]
    print(report["input"]["id"], report["result"]["status"], "blocked:", failed)
    assert verify(report) == report


## 3. Choose inputs and export the review

Choose a case ID above, or set `input_file` to a scenario JSON path. The output contains five evidence files, including input-bound Ascension verification jobs. The default output is local; set `GOVERNANCE_OUTPUT` to choose another directory.

In [ ]:
case_name = "accountable-upgrade"
input_file = None
scenario = read_json(Path(input_file)) if input_file else next(c for c in cases if c["id"] == case_name)
report = evaluate(scenario)
print(brief(report))
output = Path(os.environ.get("GOVERNANCE_OUTPUT", "governance-runs"))
folder = write_bundle(report, output)
print("Evidence:", folder.resolve())
assert verify(read_json(folder / "dossier.json")) == report
print("Replay verified. Nine jobs remain unsubmitted.")


## 4. Preserve the original simulator

Its `delta` is a numerical update rate, not the discount factor used by the review model. The two calculations address different questions. The result below is a mean-field illustration, not evidence of universal convergence.

In [ ]:
from alpha_factory_v1.demos.solving_agi_governance.governance_sim import run_sim
for update_rate in (0.2, 0.5, 0.8):
    mean = run_sim(100, 1000, update_rate, 2.5, seed=42)
    print(f"Update rate {update_rate}: mean cooperation {mean:.6f}")
